# T Cell Only Ref-Query Merge Pipeline v1.4

**Changelog v1.4 (incremental from v1.3):**
- `export_celltypist_direct_results()`: CellTypist is now a first-class output branch (adds `celltypist_label_direct`, `celltypist_label_direct_filt`, `obsm['celltypist_proba']`, `uns['celltypist_label_order']`)
- `ensure_batch_tissue()`: `batch_key` now also gets `fillna('unknown_batch')` (previously only `tissue_key` was protected; NA values crashed scVI setup)
- `run_celltypist_on_full_genes()`: removed unused `hvg_mask` parameter
- Step 6 (concat): `BATCH_KEY` values prefixed `ref_`/`qry_` to prevent sample-name collisions between datasets
- Step 18 (visualization): UMAP comparison uses safe `X_umap` swap instead of unstable `basis=` kwarg
- Clarified second scANVI branch: `celltypist_pseudolabel_refinement` (query has CellTypist labels before training — not pure semi-supervised)
- Fixed version strings in summary (was v1.2)


## Cell 0 — Imports & Global Settings

In [2]:
import sys
import warnings
import json
import gc
import joblib
from pathlib import Path
from datetime import datetime
from typing import Optional, List, Dict, Any

import numpy as np
import pandas as pd
from scipy.sparse import issparse, csr_matrix
from scipy import sparse
from scipy.stats import entropy

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os
import torch
import scanpy as sc
import scvi
import celltypist
from celltypist import models
from umap import UMAP

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

print("=" * 80)
print("T Cell ONLY Ref-Query Merge Pipeline v1.4")
print("=" * 80)

gpu_available = torch.cuda.is_available()
print(f"GPU available: {gpu_available}")
if gpu_available:
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

scvi.settings.dl_num_workers = 0
print(f"scVI dl_num_workers: {scvi.settings.dl_num_workers}")

/home/h2048/miniconda3/envs/scvi_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/h2048/miniconda3/envs/scvi_env/lib/python3.10/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


T Cell ONLY Ref-Query Merge Pipeline v1.4
GPU available: True
GPU Device: Tesla V100-SXM2-16GB
scVI dl_num_workers: 0


## Cell 1 — Configuration
> **Modify `REFERENCE_H5AD` and `QUERY_H5AD` before running**

In [3]:
# 1. CONFIGURATION
# ==============================================================================

REFERENCE_H5AD = "/home/h2048/data/py/0129/tnk_analysis_unified/results/subcluster_unified_v2_20260129/adata_tnk_subclustered_FINAL_v2_0_1_20260129.h5ad"
QUERY_H5AD     = "/home/h2048/data/py/0127/scarches_mapping_FIXED_v1_2/subsets/t_cells.h5ad"

REF_LABEL_COARSE = "cell_type_L2"
REF_LABEL_FINE   = "cell_type_L3"

BATCH_KEY  = "sample"
TISSUE_KEY = "tissue"

OUTPUT_DIR    = "/home/h2048/data/py/20260310/tcell_only_merged_pipeline"
OUTPUT_PREFIX = "tcell_only_merged"

INCLUDE_COARSE_TYPES = None

N_HVG                = 4000
FORCE_MARKERS_IN_HVG = True

SCVI_N_LATENT   = 100
SCVI_N_LAYERS   = 2
SCVI_N_HIDDEN   = 128
SCVI_DROPOUT    = 0.1
MAX_EPOCHS_SCVI = 400

MAX_EPOCHS_SCANVI  = 200
UNLABELED_CATEGORY = "Unknown"

BATCH_SIZE    = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 0.0

CELLTYPIST_MODEL         = "/home/h2048/data/source/reference/celltypist_models/Immune_All_Low.pkl"
CELLTYPIST_MAJORITY_VOTE = True

# --- CellTypist direct output branch (NEW in v1.4) ---
CELLTYPIST_DIRECT_LABEL_KEY = "celltypist_label_direct"
CELLTYPIST_DIRECT_FILT_KEY  = "celltypist_label_direct_filt"
CELLTYPIST_CONF_THRESHOLD   = 0.5
CELLTYPIST_SAVE_PROBA       = True

# --- Second scANVI branch: CellTypist pseudo-label refinement ---
# NOTE: query cells receive CellTypist labels before scANVI training,
#       so this is pseudo-label refinement, NOT pure semi-supervised transfer.
CELLTYPIST_SCANVI_USE_MAJORITY          = True
CELLTYPIST_SCANVI_LABEL_KEY             = "scanvi_labels_celltypist_pseudolabel"
CELLTYPIST_SCANVI_RESULT_KEY            = "celltypist_pseudolabel"
CELLTYPIST_SCANVI_KEEP_DP_DN_UNPREFIXED = True

QUERY_LEIDEN_RESOLUTION = 1.0

CD4_SCORE_THRESHOLD = 0.3
CD8_SCORE_THRESHOLD = 0.3

TCELL_CORE_MARKERS = ["CD3D", "CD3E", "CD3G", "PTPRC"]
CD4_MARKERS        = ["CD4", "IL7R", "CD40LG"]
CD8_MARKERS        = ["CD8A", "CD8B"]
NAIVE_MARKERS      = ["CCR7", "SELL", "TCF7", "LEF1", "CD27"]
CM_MARKERS         = ["CCR7", "CD27", "IL7R"]
EM_MARKERS         = ["GZMK", "CXCR3", "CCR5"]
TEMRA_MARKERS      = ["GZMB", "PRF1", "GNLY", "NKG7"]
TREG_MARKERS       = ["FOXP3", "IL2RA", "CTLA4", "IKZF2"]
TH1_MARKERS        = ["TBX21", "IFNG", "CXCR3"]
TH2_MARKERS        = ["GATA3", "IL4", "IL5", "IL13"]
TH17_MARKERS       = ["RORC", "IL17A", "IL17F", "CCR6"]
TRM_MARKERS        = ["CD69", "ITGAE", "CXCR6"]
PROLIF_MARKERS     = ["MKI67", "TOP2A", "PCNA"]

STRESS_SIGNATURE_GENES = [
    "HSPA1A", "HSPA1B", "HSPA8", "HSP90AA1", "HSP90AB1", "DNAJB1",
    "JUN", "JUNB", "JUND", "FOS", "FOSB", "EGR1", "IER2"
]

S_GENES = [
    "MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UHRF1",
    "GINS2","MCM6","CDCA7","DTL","PRIM1","HELLS","RFC2","RPA2",
    "NASP","RAD51AP1","GMNN","WDR76","SLBP","CCNE2","UBR7",
    "POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1"
]

G2M_GENES = [
    "HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80",
    "CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF","TACC3","FAM64A",
    "SMC4","CCNB1","CKAP2L","CKAP2","AURKB","BUB1","KIF11","ANP32E"
]

FORCED_MARKERS = list(set(
    TCELL_CORE_MARKERS + CD4_MARKERS + CD8_MARKERS +
    NAIVE_MARKERS + CM_MARKERS + EM_MARKERS + TEMRA_MARKERS +
    TREG_MARKERS + TH1_MARKERS + TH2_MARKERS + TH17_MARKERS +
    TRM_MARKERS + PROLIF_MARKERS
))

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sc.settings.seed = RANDOM_SEED
scvi.settings.seed = RANDOM_SEED

Seed set to 42


## Cell 2 — Helper Functions

In [4]:
# 2. HELPER FUNCTIONS
# ==============================================================================

def ensure_counts_layer(adata, counts_layer="counts"):
    # Robust counts validation with float32 tolerance.
    if counts_layer not in (adata.layers or {}):
        print(f"  WARNING: layers['{counts_layer}'] not found, checking .X...")
        if hasattr(adata, 'X') and adata.X is not None:
            # BUG-6 FIX: sparse[:1000] returns sub-matrix; must call .toarray().ravel()
            if issparse(adata.X):
                X_sample = adata.X[:1000].toarray().ravel()
            else:
                X_sample = np.asarray(adata.X[:1000]).ravel()
            sample = np.asarray(X_sample, dtype=np.float64)
            if np.any(sample < 0):
                raise ValueError("adata.X contains negative values - not valid raw counts!")
            if np.allclose(sample, np.round(sample), atol=1e-6):
                print(f"  -> Auto-copying .X to layers['{counts_layer}']")
                if issparse(adata.X) and not isinstance(adata.X, csr_matrix):
                    adata.layers[counts_layer] = csr_matrix(adata.X)
                else:
                    adata.layers[counts_layer] = adata.X.copy()
            else:
                raise ValueError(
                    f"CRITICAL: layers['{counts_layer}'] not found and .X is not integer counts. "
                    f"Sample range: [{sample.min():.4f}, {sample.max():.4f}]"
                )
        else:
            raise ValueError(f"CRITICAL: layers['{counts_layer}'] not found and .X is None.")

    X_counts = adata.layers[counts_layer]
    sample_data = X_counts.data[:1000] if issparse(X_counts) else X_counts.flat[:1000]
    sample = np.asarray(sample_data, dtype=np.float64)
    if np.any(sample < 0):
        raise ValueError("counts contains negative values!")
    if not np.allclose(sample, np.round(sample), atol=1e-6):
        raise ValueError(
            f"counts looks non-integer. Range: [{sample.min():.4f}, {sample.max():.4f}]"
        )
    if issparse(X_counts) and not isinstance(X_counts, csr_matrix):
        adata.layers[counts_layer] = csr_matrix(X_counts)
    return counts_layer


def ensure_batch_tissue(adata, batch_key, tissue_key):
    # FIX v1.4: batch_key now also gets fillna("unknown_batch").
    # Previously only tissue_key was protected; NA sample values cause
    # scVI setup_anndata to crash.

    # --- batch_key ---
    if batch_key not in adata.obs.columns:
        print(f"  WARNING: {batch_key} not found, creating placeholder")
        adata.obs[batch_key] = "unknown_batch"
    adata.obs[batch_key] = adata.obs[batch_key].astype("category")
    if "unknown_batch" not in adata.obs[batch_key].cat.categories:
        adata.obs[batch_key] = adata.obs[batch_key].cat.add_categories(["unknown_batch"])
    adata.obs[batch_key] = adata.obs[batch_key].fillna("unknown_batch")

    # --- tissue_key ---
    if tissue_key not in adata.obs.columns:
        print(f"  WARNING: {tissue_key} not found, creating placeholder")
        adata.obs[tissue_key] = "unknown_tissue"
    adata.obs[tissue_key] = adata.obs[tissue_key].astype("category")
    if "unknown_tissue" not in adata.obs[tissue_key].cat.categories:
        adata.obs[tissue_key] = adata.obs[tissue_key].cat.add_categories(["unknown_tissue"])
    adata.obs[tissue_key] = adata.obs[tissue_key].fillna("unknown_tissue")


def compute_module_score_efficient(adata, gene_list, score_name, counts_layer="counts"):
    # Memory-efficient module score: mean normalized expression over marker set.
    # BUG-1 FIX: replaced sc.tl.score_genes (fails with no background genes)
    # with direct mean expression on the normalized marker subset.
    genes = [g for g in gene_list if g in adata.var_names]
    if len(genes) < 3:
        adata.obs[score_name] = 0.0
        adata.obs[f"{score_name}_norm"] = 0.0
        return

    gene_idx = adata.var_names.get_indexer(genes)
    gene_idx = gene_idx[gene_idx >= 0]
    if len(gene_idx) < 3:
        adata.obs[score_name] = 0.0
        adata.obs[f"{score_name}_norm"] = 0.0
        return

    X_subset  = adata.layers[counts_layer][:, gene_idx]
    var_subset = adata.var.iloc[gene_idx].copy()
    adata_tmp  = sc.AnnData(X=X_subset.copy(), var=var_subset)
    sc.pp.normalize_total(adata_tmp, target_sum=1e4)
    sc.pp.log1p(adata_tmp)

    X_norm = adata_tmp.X
    if issparse(X_norm):
        mean_expr = np.asarray(X_norm.mean(axis=1)).flatten()
    else:
        mean_expr = np.asarray(X_norm).mean(axis=1).flatten()

    adata.obs[score_name] = mean_expr
    scores = adata.obs[score_name]
    mn, mx = float(scores.min()), float(scores.max())
    adata.obs[f"{score_name}_norm"] = (scores - mn) / (mx - mn) if mx > mn else 0.0

    del adata_tmp, X_subset
    gc.collect()


def compute_cd4_cd8_scores(adata):
    print("  -> Computing CD4/CD8 module scores...")
    compute_module_score_efficient(adata, CD4_MARKERS, "CD4_score")
    compute_module_score_efficient(adata, CD8_MARKERS, "CD8_score")
    cd4_high = adata.obs["CD4_score_norm"] > CD4_SCORE_THRESHOLD
    cd8_high = adata.obs["CD8_score_norm"] > CD8_SCORE_THRESHOLD
    classifications = []
    for i in range(len(adata)):
        c4, c8 = cd4_high.iloc[i], cd8_high.iloc[i]
        if c4 and not c8:     classifications.append("CD4_single")
        elif c8 and not c4:   classifications.append("CD8_single")
        elif c4 and c8:       classifications.append("DP")
        else:                 classifications.append("DN")
    adata.obs["cd4_cd8_by_score"] = classifications
    for lbl in ["CD4_single", "CD8_single", "DP", "DN"]:
        print(f"    {lbl}: {sum(c == lbl for c in classifications)}")


def prepare_covariates(adata):
    print("  -> Validating counts...")
    ensure_counts_layer(adata, "counts")
    print("  -> Batch/Tissue...")
    ensure_batch_tissue(adata, BATCH_KEY, TISSUE_KEY)
    print("  -> Computing signature scores...")
    if "pct_counts_mt" not in adata.obs.columns:
        adata.var["mt"] = adata.var_names.str.startswith("MT-")
        sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, layer="counts")
    compute_cd4_cd8_scores(adata)
    compute_module_score_efficient(adata, STRESS_SIGNATURE_GENES, "stress_score")
    if not all(k in adata.obs.columns for k in ["S_score", "G2M_score"]):
        s_in = [g for g in S_GENES if g in adata.var_names]
        g_in = [g for g in G2M_GENES if g in adata.var_names]
        if len(s_in) >= 5 and len(g_in) >= 5:
            cc_union = list(dict.fromkeys(s_in + g_in))
            cc_idx   = adata.var_names.get_indexer(cc_union)
            ad_tmp   = sc.AnnData(X=adata.layers["counts"][:, cc_idx].copy(),
                                  var=adata.var.iloc[cc_idx].copy())
            sc.pp.normalize_total(ad_tmp, target_sum=1e4)
            sc.pp.log1p(ad_tmp)
            sc.tl.score_genes_cell_cycle(ad_tmp, s_genes=s_in, g2m_genes=g_in)
            adata.obs["S_score"]   = ad_tmp.obs["S_score"].values
            adata.obs["G2M_score"] = ad_tmp.obs["G2M_score"].values
            adata.obs["phase"]     = ad_tmp.obs["phase"].values
            del ad_tmp; gc.collect()
        else:
            adata.obs["S_score"]   = 0.0
            adata.obs["G2M_score"] = 0.0
            adata.obs["phase"]     = "G1"


def run_celltypist_on_full_genes(adata_merged, model_name=CELLTYPIST_MODEL, majority_vote=True):
    # Run CellTypist on full gene matrix (not HVG subset).
    # BUG-2 FIX: local path check before download_models().
    # FIX v1.4: removed unused hvg_mask parameter.
    print("\n[CellTypist] Starting annotation on FULL gene matrix...")
    if os.path.isfile(model_name):
        print(f"  -> Loading model from local path: {model_name}")
        model = models.Model.load(model_name)
    else:
        model_basename = os.path.basename(model_name)
        print(f"  -> Local model not found, downloading: {model_basename}")
        models.download_models(model=model_basename)
        model = models.Model.load(model_basename)

    model_genes = set(model.features)

    # FIX v1.5: symbol_base matching so var_names_make_unique suffixes (-1, -2) do not
    # silently drop features that the model expects under their canonical name.
    if "symbol_base" in adata_merged.var.columns:
        base_to_var = {}
        for vn, sb in zip(adata_merged.var_names, adata_merged.var["symbol_base"]):
            if sb in model_genes and sb not in base_to_var:
                base_to_var[sb] = vn
        available_genes = list(base_to_var.values())
        print(f"  -> {len(available_genes)}/{len(model_genes)} model genes matched via symbol_base "
              f"(direct var_names: {sum(g in model_genes for g in adata_merged.var_names)})")
    else:
        available_genes = [g for g in adata_merged.var_names if g in model_genes]
        print(f"  -> {len(available_genes)}/{len(model_genes)} model genes matched via direct var_names")

    if len(available_genes) < 100:
        raise ValueError(f"Too few overlapping genes ({len(available_genes)}) for CellTypist!")

    adata_ct = adata_merged[:, available_genes].copy()
    if "symbol_base" in adata_ct.var.columns:
        adata_ct.var_names = pd.Index(adata_ct.var["symbol_base"].values)
        adata_ct.var_names_make_unique()
    sc.pp.normalize_total(adata_ct, target_sum=1e4)
    sc.pp.log1p(adata_ct)

    predictions = celltypist.annotate(
        adata_ct, model=model, majority_voting=majority_vote, mode='best match'
    )

    pl = predictions.predicted_labels
    if isinstance(pl, pd.DataFrame):
        pred_labels = pl.get("predicted_labels", pl.iloc[:, 0]).astype(str).values
        conf_values = predictions.probability_matrix.max(axis=1).values
        majority_labels = pl["majority_voting"].astype(str).values if majority_vote and "majority_voting" in pl.columns else None
    else:
        pred_labels     = pl.astype(str).values
        conf_values     = predictions.probability_matrix.max(axis=1).values
        majority_labels = None

    adata_merged.obs["celltypist_pred"]       = pred_labels
    adata_merged.obs["celltypist_confidence"] = conf_values
    if majority_labels is not None:
        adata_merged.obs["celltypist_majority"] = majority_labels

    print(f"  -> CellTypist done ({len(available_genes)} genes used)")
    print(pd.Series(pred_labels).value_counts().head(10))
    del adata_ct; gc.collect()
    return predictions  # returned so caller passes to export_celltypist_direct_results


def export_celltypist_direct_results(
    adata,
    predictions,
    label_key=CELLTYPIST_DIRECT_LABEL_KEY,
    filt_key=CELLTYPIST_DIRECT_FILT_KEY,
    prefer_majority=CELLTYPIST_MAJORITY_VOTE,  # FIX v1.5: direct branch uses CT's own toggle
    conf_threshold=CELLTYPIST_CONF_THRESHOLD,
    save_proba=CELLTYPIST_SAVE_PROBA,
):
    # NEW in v1.4: promote CellTypist to a first-class output branch.
    # Writes obs[label_key], obs[filt_key], obsm["celltypist_proba"],
    # uns["celltypist_label_order"], uns["celltypist_direct"].
    # Does NOT modify any previously existing obs/obsm/uns keys.
    source_key = choose_celltypist_source_key(adata, prefer_majority=prefer_majority)

    labels = adata.obs[source_key].astype(str).copy()
    adata.obs[label_key] = pd.Series(labels, index=adata.obs_names, dtype="object").astype("category")

    labels_filt = labels.copy()
    if conf_threshold is not None and "celltypist_confidence" in adata.obs.columns:
        low_conf = adata.obs["celltypist_confidence"].astype(float) < float(conf_threshold)
        labels_filt.loc[low_conf] = UNLABELED_CATEGORY
    adata.obs[filt_key] = pd.Series(labels_filt, index=adata.obs_names, dtype="object").astype("category")

    proba_obsm_key = None
    label_order_key = None
    if save_proba and hasattr(predictions, "probability_matrix"):
        proba_df = predictions.probability_matrix
        if not isinstance(proba_df, pd.DataFrame):
            proba_df = pd.DataFrame(proba_df, index=adata.obs_names)
        else:
            proba_df = proba_df.copy()
            if len(proba_df) != adata.n_obs:
                raise ValueError(
                    f"CellTypist probability matrix row count mismatch: "
                    f"{len(proba_df)} vs {adata.n_obs}"
                )
            # FIX v1.5: explicit reindex instead of in-place rename.
            # Rename only relabels; reindex reorders rows to match obs_names.
            if not proba_df.index.equals(adata.obs_names):
                try:
                    proba_df = proba_df.reindex(adata.obs_names)
                except Exception:
                    raise ValueError(
                        "Failed to align CellTypist probability matrix to adata.obs_names"
                    )
            if proba_df.isna().any().any():
                raise ValueError("NaN in CellTypist probability matrix after index alignment!")
        adata.obsm["celltypist_proba"]      = proba_df.values.astype(np.float32)
        adata.uns["celltypist_label_order"] = [str(x) for x in proba_df.columns]
        proba_obsm_key  = "celltypist_proba"
        label_order_key = "celltypist_label_order"

    adata.uns["celltypist_direct"] = {
        "source_key":         source_key,
        "label_key":          label_key,
        "filtered_label_key": filt_key,
        "confidence_key":     "celltypist_confidence",
        "probability_key":    proba_obsm_key,
        "label_order_key":    label_order_key,
        "confidence_threshold": conf_threshold,
    }

    print(f"  -> CellTypist direct branch source: {source_key}")
    print(f"  -> Direct label distribution ({label_key}):")
    print(adata.obs[label_key].value_counts().head(15))
    print(f"  -> Filtered label distribution ({filt_key}, threshold={conf_threshold}):")
    print(adata.obs[filt_key].value_counts().head(15))
    return source_key


def compute_novelty_scores(adata_merged, proba_df):
    print("  -> Computing novelty scores (entropy-based)...")
    proba_array = proba_df.values
    entropies   = entropy(proba_array + 1e-10, axis=1)
    adata_merged.obs["scanvi_entropy"] = entropies
    ent_min, ent_max = entropies.min(), entropies.max()
    adata_merged.obs["novelty_score"] = (entropies - ent_min) / (ent_max - ent_min) if ent_max > ent_min else 0.0
    query_mask   = adata_merged.obs["data_source"] == "query"
    high_novelty = (adata_merged.obs["novelty_score"] > 0.7) & query_mask
    adata_merged.obs["is_potentially_novel"] = high_novelty
    print(f"    High novelty query cells: {high_novelty.sum()}")


def run_query_only_leiden(adata_merged, resolution=QUERY_LEIDEN_RESOLUTION):
    # BUG-4 FIX: uses query_mask.values (numpy bool) when indexing obsm array.
    print(f"\n[Novelty Detection] Running query-only Leiden (resolution={resolution})...")
    query_mask  = adata_merged.obs["data_source"] == "query"
    query_cells = adata_merged.obs_names[query_mask]
    if len(query_cells) < 10:
        print("  -> Too few query cells, skipping")
        adata_merged.obs["leiden_query"] = "N/A"
        return
    X_qry = adata_merged.obsm["X_scVI"][query_mask.values]
    adata_qry_tmp = sc.AnnData(X=X_qry, obs=adata_merged.obs.loc[query_cells].copy())
    adata_qry_tmp.obsm["X_scVI"] = X_qry
    sc.pp.neighbors(adata_qry_tmp, use_rep="X_scVI", n_neighbors=30, random_state=RANDOM_SEED)
    sc.tl.leiden(adata_qry_tmp, resolution=resolution, random_state=RANDOM_SEED)
    leiden_full = pd.Series("N/A", index=adata_merged.obs_names, dtype="object")
    leiden_full.loc[query_cells] = "qry_" + adata_qry_tmp.obs["leiden"].astype(str)
    adata_merged.obs["leiden_query"] = leiden_full.values
    print(f"  -> Found {adata_qry_tmp.obs['leiden'].nunique()} query-only clusters")
    del adata_qry_tmp; gc.collect()


def ensure_label_category(adata, label_key, unlabeled_category=UNLABELED_CATEGORY):
    adata.obs[label_key] = adata.obs[label_key].astype(str).astype("category")
    if unlabeled_category not in adata.obs[label_key].cat.categories:
        adata.obs[label_key] = adata.obs[label_key].cat.add_categories([unlabeled_category])
    return label_key


def choose_celltypist_source_key(adata, prefer_majority=True):
    if prefer_majority and "celltypist_majority" in adata.obs.columns:
        return "celltypist_majority"
    if "celltypist_pred" in adata.obs.columns:
        return "celltypist_pred"
    raise KeyError("CellTypist labels not found in adata.obs")


def is_non_t_like_label(label):
    return any(kw in str(label).lower() for kw in ["nk", "natural killer", "ilc"])


def build_celltypist_pseudolabel_scanvi_labels(
    adata,
    output_key=CELLTYPIST_SCANVI_LABEL_KEY,
    prefer_majority=CELLTYPIST_SCANVI_USE_MAJORITY,
    score_key="cd4_cd8_by_score",
):
    # Build pseudo-labels for second scANVI branch by combining CellTypist
    # labels with CD4/CD8 score-based prefixes.
    if score_key not in adata.obs.columns:
        raise KeyError(f"Missing score key: {score_key}")
    source_key   = choose_celltypist_source_key(adata, prefer_majority=prefer_majority)
    base_labels  = adata.obs[source_key].astype(str)
    score_labels = adata.obs[score_key].astype(str)
    combined = []
    for base_label, score_label in zip(base_labels, score_labels):
        lc = str(base_label).strip()
        lu = lc.upper()
        if is_non_t_like_label(lc):
            combined.append(lc)
        elif score_label == "CD4_single" and "CD4" not in lu:
            combined.append(f"CD4+ {lc}")
        elif score_label == "CD8_single" and "CD8" not in lu:
            combined.append(f"CD8+ {lc}")
        elif not CELLTYPIST_SCANVI_KEEP_DP_DN_UNPREFIXED and score_label in {"DP", "DN"}:
            combined.append(f"{score_label} {lc}")
        else:
            combined.append(lc)
    adata.obs["celltypist_scanvi_source_label"]   = base_labels.values
    adata.obs["celltypist_scanvi_combined_label"] = pd.Series(combined, index=adata.obs_names, dtype="object")
    adata.obs[output_key] = adata.obs["celltypist_scanvi_combined_label"].copy()
    ensure_label_category(adata, output_key)
    print(f"  -> Pseudo-label branch source: {source_key}")
    print(f"  -> Pseudo-label distribution ({output_key}):")
    print(adata.obs[output_key].value_counts().head(15))
    return output_key, source_key


def get_scanvi_soft_predictions(scanvi_model, adata_input):
    proba_raw = scanvi_model.predict(adata_input, soft=True)
    if isinstance(proba_raw, pd.DataFrame):
        label_order = list(proba_raw.columns)
        proba       = proba_raw.values.astype(np.float32)
    else:
        proba = np.asarray(proba_raw, dtype=np.float32)
        try:
            label_order = list(
                scanvi_model.adata_manager.get_state_registry("labels").categorical_mapping
            )
        except Exception:
            label_order = [f"label_{i}" for i in range(proba.shape[1])]
    return proba, label_order


def train_scanvi_branch(scvi_model, adata_train, labels_key, branch_name, train_kwargs):
    if labels_key not in adata_train.obs.columns:
        raise KeyError(f"Missing labels_key: {labels_key}")
    ensure_label_category(adata_train, labels_key)
    print(f"\n  -> Initializing scANVI branch: {branch_name}")
    print(f"     labels_key={labels_key}, unique={adata_train.obs[labels_key].nunique()}")
    print(adata_train.obs[labels_key].value_counts().head(15))
    model = scvi.model.SCANVI.from_scvi_model(
        scvi_model, adata=adata_train,
        labels_key=labels_key, unlabeled_category=UNLABELED_CATEGORY
    )
    model.train(**train_kwargs)
    print(f"  -> scANVI complete: {branch_name}")
    return model


def export_scanvi_branch_results(
    scanvi_model, adata_train, adata_merged, result_suffix,
    labels_key, set_as_default=False, compute_novelty=False,
):
    lk  = f"X_scANVI_{result_suffix}"
    pk  = f"scanvi_pred_{result_suffix}"
    ck  = f"scanvi_confidence_{result_suffix}"
    pbk = f"scanvi_proba_{result_suffix}"
    ok  = f"scanvi_label_order_{result_suffix}"

    lat   = scanvi_model.get_latent_representation(adata_train)
    lat_df = pd.DataFrame(lat, index=adata_train.obs_names,
                          columns=[f"scANVI_{result_suffix}_{i}" for i in range(lat.shape[1])])
    lat_aligned = lat_df.reindex(adata_merged.obs_names)
    if lat_aligned.isna().any().any():
        raise ValueError(f"Missing latent for branch '{result_suffix}'")
    adata_merged.obsm[lk] = lat_aligned.values

    pred_series = pd.Series(scanvi_model.predict(adata_train), index=adata_train.obs_names)
    adata_merged.obs[pk] = pred_series.reindex(adata_merged.obs_names).values

    proba, label_order = get_scanvi_soft_predictions(scanvi_model, adata_train)
    proba_df = pd.DataFrame(proba, index=adata_train.obs_names, columns=label_order)
    proba_aligned = proba_df.reindex(adata_merged.obs_names)
    adata_merged.obsm[pbk] = proba_aligned.values
    adata_merged.obs[ck]   = proba_aligned.values.max(axis=1)
    adata_merged.uns[ok]   = list(label_order)

    adata_merged.uns[f"scanvi_branch_{result_suffix}"] = {
        "labels_key": labels_key, "latent_key": lk,
        "prediction_key": pk, "confidence_key": ck,
        "probability_key": pbk, "label_order_key": ok,
    }

    if set_as_default:
        adata_merged.obsm["X_scANVI"]         = adata_merged.obsm[lk].copy()
        adata_merged.obs["scanvi_pred"]        = adata_merged.obs[pk].values
        adata_merged.obs["scanvi_confidence"]  = adata_merged.obs[ck].values
        adata_merged.obsm["scanvi_proba"]      = adata_merged.obsm[pbk].copy()
        adata_merged.uns["scanvi_label_order"] = list(label_order)

    if compute_novelty:
        compute_novelty_scores(adata_merged, proba_aligned)

    return {"latent_key": lk, "prediction_key": pk, "confidence_key": ck,
            "probability_key": pbk, "label_order_key": ok, "label_order": label_order}

## Cell 3 — Initialization

In [5]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

## Cell 4 — Step 1: Load Reference and Query

In [6]:
print("\n[Step 1] Loading Reference and Query...")
print(f"  Loading reference: {REFERENCE_H5AD}")
adata_ref = sc.read_h5ad(REFERENCE_H5AD)
adata_ref.var_names_make_unique()
print(f"    Reference shape: {adata_ref.shape}")

print(f"  Loading query: {QUERY_H5AD}")
adata_qry = sc.read_h5ad(QUERY_H5AD)
adata_qry.var_names_make_unique()
print(f"    Query shape: {adata_qry.shape}")


[Step 1] Loading Reference and Query...
  Loading reference: /home/h2048/data/py/0129/tnk_analysis_unified/results/subcluster_unified_v2_20260129/adata_tnk_subclustered_FINAL_v2_0_1_20260129.h5ad
    Reference shape: (44833, 34252)
  Loading query: /home/h2048/data/py/0127/scarches_mapping_FIXED_v1_2/subsets/t_cells.h5ad
    Query shape: (145335, 83690)


## Cell 5 — Step 2: Subset Reference if needed

In [7]:
if INCLUDE_COARSE_TYPES is not None and REF_LABEL_COARSE in adata_ref.obs.columns:
    print(f"\n[Step 2] Subsetting reference to: {INCLUDE_COARSE_TYPES}")
    mask = adata_ref.obs[REF_LABEL_COARSE].isin(INCLUDE_COARSE_TYPES)
    adata_ref = adata_ref[mask].copy()
    print(f"    Reference after subset: {adata_ref.shape}")

## Cell 6 — Step 3: Prepare labels

In [8]:
print("\n[Step 3] Preparing labels...")
if REF_LABEL_COARSE in adata_ref.obs.columns:
    adata_ref.obs["cell_type_coarse"] = adata_ref.obs[REF_LABEL_COARSE].astype(str)
    print(f"  -> Coarse labels from: {REF_LABEL_COARSE}")
    print(adata_ref.obs["cell_type_coarse"].value_counts())
else:
    print(f"  WARNING: {REF_LABEL_COARSE} not found, using 'T_cell'")
    adata_ref.obs["cell_type_coarse"] = "T_cell"

if REF_LABEL_FINE in adata_ref.obs.columns:
    adata_ref.obs["cell_type_fine"] = adata_ref.obs[REF_LABEL_FINE].astype(str)
    print(f"  -> Fine labels from: {REF_LABEL_FINE}")
else:
    adata_ref.obs["cell_type_fine"] = adata_ref.obs["cell_type_coarse"]
    print("  -> Using coarse labels as fine labels")

adata_qry.obs["cell_type_coarse"] = UNLABELED_CATEGORY
adata_qry.obs["cell_type_fine"]   = UNLABELED_CATEGORY


[Step 3] Preparing labels...
  -> Coarse labels from: cell_type_L2
cell_type_coarse
CD16+ NK cells                      10963
Trm cytotoxic T cells                9500
CD8+ Trm cytotoxic T cells           5412
Tem/Effector helper T cells          3845
Tem/Trm cytotoxic T cells            3652
NK cells                             2847
CD8+ Tem/Trm cytotoxic T cells       2123
Tem/Temra cytotoxic T cells          1705
CD8+ Tem/Temra cytotoxic T cells     1642
Regulatory T cells                    967
Type 17 helper T cells                335
MAIT cells                            298
Follicular helper T cells             253
CD8+ Tem/Effector helper T cells      216
CD8+ gamma-delta T cells              194
ILC3                                  180
Tcm/Naive helper T cells              142
CD4+ Regulatory T cells               140
gamma-delta T cells                   131
Type 1 helper T cells                 104
CD16- NK cells                         94
CD4+ Tem/Effector helper T cells 

## Cell 7 — Step 4: Find common genes

In [9]:
print("\n[Step 4] Finding common genes...")
qry_set      = set(adata_qry.var_names)
common_genes = [g for g in adata_ref.var_names if g in qry_set]
print(f"  Reference: {adata_ref.n_vars:,}, Query: {adata_qry.n_vars:,}, Common: {len(common_genes):,}")
if len(common_genes) < 1000:
    raise ValueError(f"Too few common genes ({len(common_genes)}). Check gene naming!")
adata_ref = adata_ref[:, common_genes].copy()
adata_qry = adata_qry[:, common_genes].copy()


[Step 4] Finding common genes...
  Reference: 34,252, Query: 83,690, Common: 33,749


## Cell 8 — Step 5: Validate counts

In [10]:
print("\n[Step 5] Validating counts...")
ensure_counts_layer(adata_ref, "counts")
ensure_counts_layer(adata_qry, "counts")


[Step 5] Validating counts...
  -> Auto-copying .X to layers['counts']


'counts'

## Cell 9 — Step 6: Concatenate (with batch-prefix fix)

In [11]:
print("\n[Step 6] Concatenating...")

# FIX v1.4: Prefix BATCH_KEY values with ref_/qry_ to prevent sample-name
# collisions (e.g., both datasets may have sample="P1").
# FIX v1.5: check column existence before prefixing; use astype("string").fillna()
# so NA values become "unknown_batch" not the literal string "nan".
for ad, prefix in [(adata_ref, "ref_"), (adata_qry, "qry_")]:
    if BATCH_KEY not in ad.obs.columns:
        print(f"  WARNING: {BATCH_KEY} not found before concat, creating placeholder")
        ad.obs[BATCH_KEY] = "unknown_batch"
    batch_vals = ad.obs[BATCH_KEY].astype("string").fillna("unknown_batch")
    ad.obs[BATCH_KEY] = prefix + batch_vals

adata_ref.obs_names = pd.Index([f"ref_{x}" for x in adata_ref.obs_names])
adata_qry.obs_names = pd.Index([f"qry_{x}" for x in adata_qry.obs_names])

adata_merged = sc.concat(
    {"reference": adata_ref, "query": adata_qry},
    axis=0, join="inner", merge="unique", label="data_source"
)

print(f"  Merged: {adata_merged.shape}")
print(f"  Reference: {(adata_merged.obs['data_source'] == 'reference').sum():,}")
print(f"  Query: {(adata_merged.obs['data_source'] == 'query').sum():,}")

del adata_ref, adata_qry
gc.collect()


[Step 6] Concatenating...
  Merged: (190168, 33749)
  Reference: 44,833
  Query: 145,335


47547

## Cell 10 — Step 7: Prepare covariates

In [12]:
print("\n[Step 7] Preparing covariates...")
prepare_covariates(adata_merged)
print("  -> Adding symbol_base column...")
adata_merged.var["symbol_base"] = adata_merged.var_names.str.replace(r"-\d+$", "", regex=True)


[Step 7] Preparing covariates...
  -> Validating counts...
  -> Batch/Tissue...
  -> Computing signature scores...
  -> Computing CD4/CD8 module scores...
    CD4_single: 96750
    CD8_single: 0
    DP: 0
    DN: 93418
  -> Adding symbol_base column...


## Cell 11 — Step 8: HVG Selection

In [13]:
print("\n[Step 8] Selecting HVGs...")
hvg_method = "unknown"
try:
    sc.pp.highly_variable_genes(
        adata_merged, layer="counts", n_top_genes=N_HVG,
        batch_key=BATCH_KEY, flavor="seurat_v3", subset=False
    )
    hvg_method = "batch_seurat_v3"
except Exception as e1:
    print(f"  -> batch-aware failed ({str(e1)[:50]}), trying standard...")
    try:
        sc.pp.highly_variable_genes(
            adata_merged, layer="counts", n_top_genes=N_HVG,
            flavor="seurat_v3", subset=False
        )
        hvg_method = "standard_seurat_v3"
    except Exception as e2:
        print(f"  -> standard failed ({str(e2)[:50]}), fallback to cell_ranger")
        sc.pp.highly_variable_genes(
            adata_merged, layer="counts", n_top_genes=N_HVG,
            flavor="cell_ranger", subset=False
        )
        hvg_method = "cell_ranger"

print(f"  -> Method: {hvg_method}")

if FORCE_MARKERS_IN_HVG:
    n_added    = 0
    marker_set = set(FORCED_MARKERS)
    for idx, symbol_base in enumerate(adata_merged.var["symbol_base"]):
        if symbol_base in marker_set:
            rn = adata_merged.var_names[idx]
            if not adata_merged.var.loc[rn, "highly_variable"]:
                adata_merged.var.loc[rn, "highly_variable"] = True
                n_added += 1
    print(f"  -> Forced {n_added}/{len(FORCED_MARKERS)} markers into HVG")

n_hvg_final = adata_merged.var["highly_variable"].sum()
print(f"  -> Final HVG count: {n_hvg_final}")

hvg_genes = adata_merged.var_names[adata_merged.var["highly_variable"]].tolist()
with open(output_dir / f"{OUTPUT_PREFIX}_hvg_genes.txt", "w") as f:
    f.write("\n".join(hvg_genes))


[Step 8] Selecting HVGs...
  -> batch-aware failed (b'There are other near singularities as well. 0.09), trying standard...
  -> Method: standard_seurat_v3
  -> Forced 13/41 markers into HVG
  -> Final HVG count: 4013


## Cell 12 — Step 9: Build FULL matrix for .raw

In [14]:
print("\n[Step 9] Building full matrix for .raw...")
full_counts = adata_merged.layers["counts"]
if issparse(full_counts) and not isinstance(full_counts, csr_matrix):
    full_counts = csr_matrix(full_counts)
raw_var = adata_merged.var.copy()
print(f"  Full matrix shape: {full_counts.shape}")


[Step 9] Building full matrix for .raw...
  Full matrix shape: (190168, 33749)


## Cell 13 — Step 10: Create training subset

In [15]:
print("\n[Step 10] Creating training subset...")

hvg_mask = adata_merged.var["highly_variable"].values
X_hvg    = adata_merged.layers["counts"][:, hvg_mask]
if issparse(X_hvg) and not isinstance(X_hvg, csr_matrix):
    X_hvg = csr_matrix(X_hvg)

adata_train = sc.AnnData(
    X=X_hvg.copy(),
    obs=adata_merged.obs.copy(),
    var=adata_merged.var.iloc[hvg_mask].copy()
)
adata_train.var_names      = adata_merged.var_names[hvg_mask]
adata_train.layers["counts"] = adata_train.X

print(f"  Training data: {adata_train.shape}")

adata_train.obs["scanvi_labels"] = adata_train.obs["cell_type_fine"].astype(str)
ensure_label_category(adata_train, "scanvi_labels")

# Placeholder for pseudo-label branch; overwritten after CellTypist in Step 11.
adata_train.obs[CELLTYPIST_SCANVI_LABEL_KEY] = adata_train.obs["scanvi_labels"].astype(str)
ensure_label_category(adata_train, CELLTYPIST_SCANVI_LABEL_KEY)

print("  -> Reference-label distribution:")
print(adata_train.obs["scanvi_labels"].value_counts())
gc.collect()


[Step 10] Creating training subset...
  Training data: (190168, 4013)
  -> Reference-label distribution:
scanvi_labels
Unknown                                   145335
cd16plus_nk_cells_c0                        5023
cd16plus_nk_cells_c1                        4215
trm_cytotoxic_t_cells_c0                    4058
trm_cytotoxic_t_cells_c1                    2993
cd8plus_trm_cytotoxic_t_cells_c0            2509
trm_cytotoxic_t_cells_c2                    2449
tem_effector_helper_t_cells_c0              2005
tem_trm_cytotoxic_t_cells_c0                1932
tem_effector_helper_t_cells_c1              1840
cd16plus_nk_cells_c2                        1725
tem_trm_cytotoxic_t_cells_c1                1720
nk_cells_c0                                 1590
cd8plus_trm_cytotoxic_t_cells_c1            1569
cd8plus_tem_trm_cytotoxic_t_cells_c0        1337
cd8plus_trm_cytotoxic_t_cells_c2            1334
nk_cells_c1                                 1257
tem_temra_cytotoxic_t_cells_c0               98

20

## Cell 13b — Checkpoint: AnnData structure log

In [16]:
print("\n[Checkpoint] AnnData structure before CellTypist...")

def _brief_summary(adata, name):
    lines = [f"[{name}] shape={adata.shape}",
             f"  obs: {list(adata.obs.columns)}",
             f"  layers: {list(adata.layers.keys())}",
             f"  obsm: {list(adata.obsm.keys())}",
             f"  raw present: {adata.raw is not None}"]
    return "\n".join(lines)

for n, a in [("adata_merged", adata_merged), ("adata_train", adata_train)]:
    print(_brief_summary(a, n))

log_path = output_dir / f"{OUTPUT_PREFIX}_anndata_structure.log"
with open(log_path, "a", encoding="utf-8") as f:
    ts = datetime.now().isoformat(timespec='seconds')
    f.write(f"\n=== {ts} :: pre-CellTypist checkpoint ===\n")
    for n, a in [("adata_merged", adata_merged), ("adata_train", adata_train)]:
        f.write(_brief_summary(a, n) + "\n")
print(f"  -> Checkpoint log: {log_path}")


[Checkpoint] AnnData structure before CellTypist...
[adata_merged] shape=(190168, 33749)
  obs: ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'age', 'sex', 'percent.mt', 'tissue', 'tissue_sampling_method', 'dataset', 'sample', 'percent.rb', 'decontX_contamination', 'donor_id', 'Group', 'Ethnicity_inferred', 'Smoker', 'COVID_status', 'First_symptoms_collection_interval', 'Kit_version', 'batch', 'log1p_n_genes', 'percent_total_sarscov2', 'n_counts_sarscov2', 'scrublet_score', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'reference_genome', 'tissue_dissociation_protocol', 'tissue_level_2', 'cell_id', 'condition', 'doublet_class', 'barcode', '_scvi_batch', '_scvi_labels', 'cell_type_coarse', 'cell_type_fine', 'data_source', 

## Cell 14 — Step 11: CellTypist (full genes) + Direct Export (NEW v1.4)

In [17]:
# STEP_BREAK: c14_celltypist
print("\n[Step 11] Running CellTypist on full gene matrix...")
try:
    # run_celltypist_on_full_genes now returns the predictions object (v1.4)
    celltypist_predictions = run_celltypist_on_full_genes(adata_merged)

    # --- NEW in v1.4: promote CellTypist to first-class output branch ---
    direct_source_key = export_celltypist_direct_results(
        adata_merged,
        celltypist_predictions,
        label_key=CELLTYPIST_DIRECT_LABEL_KEY,
        filt_key=CELLTYPIST_DIRECT_FILT_KEY,
        prefer_majority=CELLTYPIST_MAJORITY_VOTE,  # FIX v1.5: direct branch switch
        conf_threshold=CELLTYPIST_CONF_THRESHOLD,
        save_proba=CELLTYPIST_SAVE_PROBA,
    )

    # Build pseudo-label branch labels (CellTypist + CD4/CD8 score prefix)
    alt_label_key, alt_source_key = build_celltypist_pseudolabel_scanvi_labels(
        adata_merged,
        output_key=CELLTYPIST_SCANVI_LABEL_KEY,
        prefer_majority=CELLTYPIST_SCANVI_USE_MAJORITY,
    )

    # Back-fill all CellTypist-derived columns to adata_train
    cols_to_backfill = [
        "celltypist_pred",
        "celltypist_confidence",
        CELLTYPIST_DIRECT_LABEL_KEY,
        CELLTYPIST_DIRECT_FILT_KEY,
        "celltypist_scanvi_source_label",
        "celltypist_scanvi_combined_label",
        alt_label_key,
    ]
    if "celltypist_majority" in adata_merged.obs.columns:
        cols_to_backfill.append("celltypist_majority")

    for col in cols_to_backfill:
        if col in adata_merged.obs.columns:
            adata_train.obs[col] = adata_merged.obs.loc[adata_train.obs_names, col].values

    ensure_label_category(adata_train, alt_label_key)
    print(f"  -> Pseudo-label scANVI labels ready from: {alt_source_key}")
    print(adata_train.obs[alt_label_key].value_counts().head(15))

except Exception as e:
    print(f"  WARNING: CellTypist failed: {e}")
    import traceback; traceback.print_exc()
    print("  -> Falling back to reference fine labels for pseudo-label branch")
    adata_merged.obs[CELLTYPIST_SCANVI_LABEL_KEY] = adata_merged.obs["cell_type_fine"].astype(str)
    ensure_label_category(adata_merged, CELLTYPIST_SCANVI_LABEL_KEY)
    adata_train.obs[CELLTYPIST_SCANVI_LABEL_KEY]  = adata_train.obs["scanvi_labels"].astype(str)
    ensure_label_category(adata_train, CELLTYPIST_SCANVI_LABEL_KEY)


[Step 11] Running CellTypist on full gene matrix...

[CellTypist] Starting annotation on FULL gene matrix...
  -> Loading model from local path: /home/h2048/data/source/reference/celltypist_models/Immune_All_Low.pkl
  -> 6088/6639 model genes matched via symbol_base (direct var_names: 6236)


🔬 Input data has 190168 cells and 6088 genes
🔗 Matching reference genes in the model
🧬 6088 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
⛓️ Over-clustering input data with resolution set to 25
🗳️ Majority voting the predictions
✅ Majority voting done!


  -> CellTypist done (6088 genes used)
Tem/Trm cytotoxic T cells      34447
Trm cytotoxic T cells          25678
Regulatory T cells             24160
CD16+ NK cells                 17570
Tem/Effector helper T cells    15675
Tem/Temra cytotoxic T cells    13759
CD16- NK cells                  8353
Tcm/Naive helper T cells        7819
Type 1 helper T cells           6764
NK cells                        6490
Name: count, dtype: int64
  -> CellTypist direct branch source: celltypist_majority
  -> Direct label distribution (celltypist_label_direct):
celltypist_label_direct
Trm cytotoxic T cells          43572
Tem/Trm cytotoxic T cells      36059
Regulatory T cells             23107
Tem/Effector helper T cells    19771
CD16+ NK cells                 17578
Tem/Temra cytotoxic T cells    12952
Tcm/Naive helper T cells       10893
CD16- NK cells                  8343
Type 17 helper T cells          3778
NK cells                        3518
Type 1 helper T cells           3329
CRTAM+ gamma-delta

## Cell 15 — Step 12: scVI Training

In [18]:
print("\n[Step 12] Training scVI...")

setup_kwargs = {
    "layer":                     "counts",
    "batch_key":                 BATCH_KEY,
    "continuous_covariate_keys": ["pct_counts_mt", "stress_score", "S_score", "G2M_score"],
    "categorical_covariate_keys": [TISSUE_KEY]
}
scvi.model.SCVI.setup_anndata(adata_train, **setup_kwargs)

scvi_model = scvi.model.SCVI(
    adata_train,
    n_latent=SCVI_N_LATENT, n_layers=SCVI_N_LAYERS,
    n_hidden=SCVI_N_HIDDEN, dropout_rate=SCVI_DROPOUT
)

train_kwargs = {
    "max_epochs": MAX_EPOCHS_SCVI, "batch_size": BATCH_SIZE,
    "early_stopping": True, "early_stopping_patience": 30,
    "plan_kwargs": {"lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
}
if gpu_available:
    train_kwargs["accelerator"] = "gpu"
    train_kwargs["devices"]     = 1

scvi_model.train(**train_kwargs)
print("  -> scVI complete")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



[Step 12] Training scVI...
Epoch 400/400: 100%|██████████| 400/400 [2:34:05<00:00, 21.76s/it, v_num=1, train_loss_step=712, train_loss_epoch=659]  

`Trainer.fit` stopped: `max_epochs=400` reached.


Epoch 400/400: 100%|██████████| 400/400 [2:34:05<00:00, 23.11s/it, v_num=1, train_loss_step=712, train_loss_epoch=659]
  -> scVI complete


## Cell 16 — Step 13: scANVI Training (dual branches)

In [19]:
print("\n[Step 13] Training scANVI branches...")

scanvi_train_kwargs = {
    "max_epochs": MAX_EPOCHS_SCANVI, "batch_size": BATCH_SIZE,
    "early_stopping": True, "early_stopping_patience": 20,
    "plan_kwargs": {"lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
}
if gpu_available:
    scanvi_train_kwargs["accelerator"] = "gpu"
    scanvi_train_kwargs["devices"]     = 1

scanvi_models = {}

# Branch 1: reference fine labels (semi-supervised; query = Unknown)
scanvi_models["reference"] = train_scanvi_branch(
    scvi_model, adata_train,
    labels_key="scanvi_labels",
    branch_name="reference_fine_labels",
    train_kwargs=scanvi_train_kwargs,
)
scanvi_model = scanvi_models["reference"]

# Branch 2: CellTypist pseudo-label refinement
# NOTE: query cells already have CellTypist labels here -> this branch is
#       pseudo-label refinement, NOT strictly semi-supervised transfer.
scanvi_models[CELLTYPIST_SCANVI_RESULT_KEY] = train_scanvi_branch(
    scvi_model, adata_train,
    labels_key=CELLTYPIST_SCANVI_LABEL_KEY,
    branch_name="celltypist_pseudolabel_refinement",
    train_kwargs=scanvi_train_kwargs,
)
scanvi_model_pseudolabel = scanvi_models[CELLTYPIST_SCANVI_RESULT_KEY]

print("  -> Dual scANVI complete")


[Step 13] Training scANVI branches...

  -> Initializing scANVI branch: reference_fine_labels
     labels_key=scanvi_labels, unique=39
scanvi_labels
Unknown                                 145335
cd16plus_nk_cells_c0                      5023
cd16plus_nk_cells_c1                      4215
trm_cytotoxic_t_cells_c0                  4058
trm_cytotoxic_t_cells_c1                  2993
cd8plus_trm_cytotoxic_t_cells_c0          2509
trm_cytotoxic_t_cells_c2                  2449
tem_effector_helper_t_cells_c0            2005
tem_trm_cytotoxic_t_cells_c0              1932
tem_effector_helper_t_cells_c1            1840
cd16plus_nk_cells_c2                      1725
tem_trm_cytotoxic_t_cells_c1              1720
nk_cells_c0                               1590
cd8plus_trm_cytotoxic_t_cells_c1          1569
cd8plus_tem_trm_cytotoxic_t_cells_c0      1337
Name: count, dtype: int64
INFO     Training for 200 epochs.                                                                                  


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 200/200: 100%|██████████| 200/200 [2:51:05<00:00, 51.27s/it, v_num=1, train_loss_step=669, train_loss_epoch=654]  

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 200/200: 100%|██████████| 200/200 [2:51:05<00:00, 51.33s/it, v_num=1, train_loss_step=669, train_loss_epoch=654]
  -> scANVI complete: reference_fine_labels

  -> Initializing scANVI branch: celltypist_pseudolabel_refinement
     labels_key=scanvi_labels_celltypist_pseudolabel, unique=40
scanvi_labels_celltypist_pseudolabel
Trm cytotoxic T cells               22767
CD4+ Trm cytotoxic T cells          20805
Tem/Trm cytotoxic T cells           19775
CD16+ NK cells                      17578
CD4+ Tem/Effector helper T cells    17359
CD4+ Regulatory T cells             17337
CD4+ Tem/Trm cytotoxic T cells      16284
Tem/Temra cytotoxic T cells          8723
CD16- NK cells                       8343
CD4+ Tcm/Naive helper T cells        5994
Regulatory T cells                   5770
Tcm/Naive helper T cells             4899
CD4+ Tem/Temra cytotoxic T cells     4229
NK cells                             3518
CD4+ Type 17 helper T cells          3184
Name: count, dtype: int64
INFO     Tra

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 200/200: 100%|██████████| 200/200 [2:47:56<00:00, 48.93s/it, v_num=1, train_loss_step=632, train_loss_epoch=658]  

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 200/200: 100%|██████████| 200/200 [2:47:56<00:00, 50.38s/it, v_num=1, train_loss_step=632, train_loss_epoch=658]
  -> scANVI complete: celltypist_pseudolabel_refinement
  -> Dual scANVI complete


## Cell 17 — Step 14: Export scANVI Results

In [20]:
print("\n[Step 14] Exporting scANVI Results...")

scanvi_branch_results = {}

scanvi_branch_results["reference"] = export_scanvi_branch_results(
    scanvi_model, adata_train, adata_merged,
    result_suffix="reference",
    labels_key="scanvi_labels",
    set_as_default=True,
    compute_novelty=True,
)

scanvi_branch_results[CELLTYPIST_SCANVI_RESULT_KEY] = export_scanvi_branch_results(
    scanvi_model_pseudolabel, adata_train, adata_merged,
    result_suffix=CELLTYPIST_SCANVI_RESULT_KEY,
    labels_key=CELLTYPIST_SCANVI_LABEL_KEY,
    set_as_default=False,
    compute_novelty=False,
)

print("  -> Branch 1 (reference-label) predictions:")
print(adata_merged.obs["scanvi_pred_reference"].value_counts().head(15))
print(f"\n  -> Branch 2 (pseudo-label) predictions:")
print(adata_merged.obs[f"scanvi_pred_{CELLTYPIST_SCANVI_RESULT_KEY}"].value_counts().head(15))


[Step 14] Exporting scANVI Results...
  -> Computing novelty scores (entropy-based)...
    High novelty query cells: 668
  -> Branch 1 (reference-label) predictions:
scanvi_pred_reference
type_17_helper_t_cells_c0                 147754
cd16plus_nk_cells_c0                       29478
regulatory_t_cells_c1                       7991
regulatory_t_cells_c0                       1765
cd16plus_nk_cells_c1                        1270
ilc3_c0                                      606
cd16-_nk_cells_c0                            548
nk_cells_c0                                  192
tem_effector_helper_t_cells_c1               107
tem_trm_cytotoxic_t_cells_c1                  90
cd8plus_trm_cytotoxic_t_cells_c0              70
cd8plus_tem_temra_cytotoxic_t_cells_c1        69
cd8plus_tem_effector_helper_t_cells_c0        48
cd8plus_tem_trm_cytotoxic_t_cells_c0          46
trm_cytotoxic_t_cells_c2                      44
Name: count, dtype: int64

  -> Branch 2 (pseudo-label) predictions:
scanvi_

## Cell 18 — Step 15: Attach .raw

In [21]:
print("\n[Step 15] Attaching .raw...")
from anndata import AnnData
adata_merged.raw = AnnData(X=full_counts, obs=adata_merged.obs.copy(), var=raw_var)
print(f"  OK .raw: {adata_merged.raw.n_vars} genes")


[Step 15] Attaching .raw...
  OK .raw: 33749 genes


## Cell 19 — Step 16: Compute UMAPs (scVI + scANVI)

In [22]:
print("\n[Step 16] Computing Multiple UMAPs...")

# scVI latent (BUG-3 FIX: index-aligned reindex)
latent_scvi = scvi_model.get_latent_representation(adata_train)
lat_df      = pd.DataFrame(
    latent_scvi, index=adata_train.obs_names,
    columns=[f"scVI_{i}" for i in range(latent_scvi.shape[1])]
)
lat_aligned = lat_df.reindex(adata_merged.obs_names)
if lat_aligned.isna().any().any():
    raise ValueError("CRITICAL: Missing scVI latent representation after reindex!")
adata_merged.obsm["X_scVI"] = lat_aligned.values

run_query_only_leiden(adata_merged, resolution=QUERY_LEIDEN_RESOLUTION)

# 16a: UMAP on scVI latent
print("  -> Computing UMAP on scVI latent...")
sc.pp.neighbors(adata_merged, use_rep="X_scVI",   n_neighbors=30, random_state=RANDOM_SEED, key_added="neighbors_scVI")
sc.tl.umap(adata_merged, random_state=RANDOM_SEED, neighbors_key="neighbors_scVI")
adata_merged.obsm["X_umap_scVI"] = adata_merged.obsm["X_umap"].copy()

# 16b: UMAP on scANVI latent (DEFAULT)
print("  -> Computing UMAP on scANVI latent (DEFAULT)...")
sc.pp.neighbors(adata_merged, use_rep="X_scANVI", n_neighbors=30, random_state=RANDOM_SEED, key_added="neighbors_scANVI")
sc.tl.umap(adata_merged, random_state=RANDOM_SEED, neighbors_key="neighbors_scANVI")
adata_merged.obsm["X_umap_scANVI"] = adata_merged.obsm["X_umap"].copy()
adata_merged.obsm["X_umap"]        = adata_merged.obsm["X_umap_scANVI"].copy()
print("     X_umap_scVI, X_umap_scANVI, X_umap (default=scANVI) all saved")

umap_op = UMAP(n_neighbors=30, n_components=2, min_dist=0.5, spread=1.0,
               metric="euclidean", random_state=RANDOM_SEED)
umap_op.fit(adata_merged.obsm["X_scANVI"])
joblib.dump(umap_op, output_dir / f"{OUTPUT_PREFIX}_umap_scanvi_operator.joblib")
print("  -> UMAP operator (scANVI) saved")

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(adata_merged.obs["scanvi_confidence"], bins=50, edgecolor="black")
ax.set_xlabel("Prediction Confidence"); ax.set_ylabel("Cell Count")
ax.set_title("scANVI Confidence Distribution")
plt.tight_layout()
plt.savefig(output_dir / f"{OUTPUT_PREFIX}_confidence_histogram.png", dpi=150)
plt.close()


[Step 16] Computing Multiple UMAPs...

[Novelty Detection] Running query-only Leiden (resolution=1.0)...
  -> Found 17 query-only clusters
  -> Computing UMAP on scVI latent...
  -> Computing UMAP on scANVI latent (DEFAULT)...
     X_umap_scVI, X_umap_scANVI, X_umap (default=scANVI) all saved
  -> UMAP operator (scANVI) saved


## Cell 20 — Step 17: Save Results

In [23]:
print("\n[Step 17] Saving Results...")

# FIX v1.5: convert label columns to category before write_h5ad.
_categorical_cols = [
    "data_source", "cell_type_coarse", "cell_type_fine",
    "scanvi_labels", "scanvi_pred", "cd4_cd8_by_score", "leiden_query",
    CELLTYPIST_DIRECT_LABEL_KEY, CELLTYPIST_DIRECT_FILT_KEY,
    CELLTYPIST_SCANVI_LABEL_KEY,
    f"scanvi_pred_{CELLTYPIST_SCANVI_RESULT_KEY}",
]
for _col in _categorical_cols:
    for _ad in [adata_merged, adata_train]:
        if _col in _ad.obs.columns:
            _ad.obs[_col] = _ad.obs[_col].astype("category")

scanvi_model.save(output_dir / f"{OUTPUT_PREFIX}_scanvi_model",           overwrite=True)
scanvi_model.save(output_dir / f"{OUTPUT_PREFIX}_scanvi_model_reference", overwrite=True)
scanvi_model_pseudolabel.save(
    output_dir / f"{OUTPUT_PREFIX}_scanvi_model_{CELLTYPIST_SCANVI_RESULT_KEY}",
    overwrite=True,
)
scvi_model.save(output_dir / f"{OUTPUT_PREFIX}_scvi_model", overwrite=True)

config = {
    "version":   "1.5_dual_scanvi",
    "timestamp": datetime.now().isoformat(),
    "input":     {"reference": REFERENCE_H5AD, "query": QUERY_H5AD},
    "n_hvg":     int(n_hvg_final),
    "annotation_layers": {
        "celltypist_direct": {
            "label_key":            CELLTYPIST_DIRECT_LABEL_KEY,
            "filtered_label_key":   CELLTYPIST_DIRECT_FILT_KEY,
            "confidence_key":       "celltypist_confidence",
            "probability_key":      "celltypist_proba",
            "label_order_key":      "celltypist_label_order",
            "confidence_threshold": CELLTYPIST_CONF_THRESHOLD,
            "description":          "Pure CellTypist output on all merged cells",
        },
        "scanvi_reference": {
            "labels_key":      "scanvi_labels",
            "prediction_key":  "scanvi_pred_reference",
            "confidence_key":  "scanvi_confidence_reference",
            "latent_key":      "X_scANVI_reference",
            "default_aliases": ["scanvi_pred", "scanvi_confidence", "X_scANVI"],
            "description":     "Reference-supervised scANVI; query = Unknown",
        },
        CELLTYPIST_SCANVI_RESULT_KEY: {
            "labels_key":    CELLTYPIST_SCANVI_LABEL_KEY,
            "prediction_key": f"scanvi_pred_{CELLTYPIST_SCANVI_RESULT_KEY}",
            "confidence_key": f"scanvi_confidence_{CELLTYPIST_SCANVI_RESULT_KEY}",
            "latent_key":     f"X_scANVI_{CELLTYPIST_SCANVI_RESULT_KEY}",
            "description":    "CellTypist pseudo-label refinement; query gets CellTypist labels before training",
        },
    },
    "umap_spaces": {
        "X_umap":        "DEFAULT - scANVI-based UMAP",
        "X_umap_scVI":   "scVI latent space UMAP",
        "X_umap_scANVI": "scANVI latent space UMAP",
    },
    "bug_fixes_v1_4": {
        "BUG1_P0":  "score_genes -> direct mean-expression (no background-gene error)",
        "BUG2_P0":  "celltypist model load: local-path check before download_models",
        "BUG3_P1":  "X_scVI: pandas reindex for index-aligned writeback",
        "BUG4_P1":  "query_mask.values for safe numpy obsm indexing",
        "BUG5_P1":  "predict(soft=True) wrapped in np.asarray() for version safety",
        "BUG6_P0":  "sparse[:1000].toarray().ravel() for integer-count validation",
        "BUG7_P1":  "ensure_batch_tissue: batch_key now also fillna('unknown_batch')",
        "BUG8_P1":  "BATCH_KEY values prefixed ref_/qry_ before concat (prevents collision)",
        "BUG9_P0":  "export_celltypist_direct_results: CellTypist is now first-class output",
        "BUG10_P1": "UMAP comparison: X_umap swap (stable) instead of basis= kwarg",
        "BUG11_P2": "Removed unused hvg_mask param from run_celltypist_on_full_genes",
    },
}

with open(output_dir / f"{OUTPUT_PREFIX}_config.json", "w") as f:
    json.dump(config, f, indent=2)

output_h5ad = output_dir / f"{OUTPUT_PREFIX}_results.h5ad"
adata_merged.write_h5ad(output_h5ad, compression="gzip")
print(f"  -> Saved: {output_h5ad}")

train_h5ad = output_dir / f"{OUTPUT_PREFIX}_train_HVG.h5ad"
adata_train.write_h5ad(train_h5ad, compression="gzip")
print(f"  -> Saved: {train_h5ad}")


[Step 17] Saving Results...


... storing 'phase' as categorical
... storing 'celltypist_pred' as categorical
... storing 'celltypist_majority' as categorical
... storing 'celltypist_scanvi_source_label' as categorical
... storing 'celltypist_scanvi_combined_label' as categorical
... storing 'scanvi_pred_reference' as categorical
... storing 'symbol_base' as categorical
... storing 'symbol_base' as categorical


  -> Saved: /home/h2048/data/py/20260310/tcell_only_merged_pipeline/tcell_only_merged_results.h5ad


... storing 'phase' as categorical
... storing 'celltypist_pred' as categorical
... storing 'celltypist_scanvi_source_label' as categorical
... storing 'celltypist_scanvi_combined_label' as categorical
... storing 'celltypist_majority' as categorical
... storing 'symbol_base' as categorical


  -> Saved: /home/h2048/data/py/20260310/tcell_only_merged_pipeline/tcell_only_merged_train_HVG.h5ad


## Cell 20b — Step 17b: Run Log + AnnData Structure

In [24]:
print("\n[Step 17b] Writing run log and AnnData structure summary...")


def summarize_anndata_full(adata, name):
    lines = ["=" * 100, f"AnnData summary: {name}", "=" * 100,
             f"shape: {adata.n_obs:,} obs x {adata.n_vars:,} vars",
             f"obs columns ({adata.obs.shape[1]}): {list(adata.obs.columns)}"]
    layer_keys = list(adata.layers.keys())
    lines.append(f"layers ({len(layer_keys)}): {layer_keys}")
    for k in layer_keys:
        lyr = adata.layers[k]
        lines.append(f"  - layers[{k!r}]: shape={getattr(lyr,'shape','NA')}, dtype={getattr(lyr,'dtype','NA')}")
    obsm_keys = list(adata.obsm.keys())
    lines.append(f"obsm ({len(obsm_keys)}): {obsm_keys}")
    for k in obsm_keys:
        lines.append(f"  - obsm[{k!r}]: shape={getattr(adata.obsm[k],'shape','NA')}")
    lines.append(f"obsp: {list(adata.obsp.keys())}")
    lines.append(f"uns keys ({len(adata.uns)}): {list(adata.uns.keys())[:40]}")
    lines.append(f"raw: {'present, shape=' + str(adata.raw.shape) if adata.raw is not None else 'None'}")
    return "\n".join(lines)


def summarize_h5ad_on_disk(h5ad_path, name):
    h5ad_path = Path(h5ad_path)
    lines = ["-" * 100, f"Saved h5ad: {name}", f"path: {h5ad_path}"]
    if not h5ad_path.exists():
        lines.append("status: missing"); return "\n".join(lines)
    lines.append(f"size_bytes: {h5ad_path.stat().st_size:,}")
    backed = sc.read_h5ad(h5ad_path, backed="r")
    try:
        lines += [f"shape: {backed.n_obs:,} obs x {backed.n_vars:,} vars",
                  f"obs columns ({backed.obs.shape[1]}): {list(backed.obs.columns)}",
                  f"layers: {list(backed.layers.keys())}",
                  f"obsm: {list(backed.obsm.keys())}",
                  f"uns keys: {list(backed.uns.keys())[:40]}",
                  f"raw present: {backed.raw is not None}"]
    finally:
        if getattr(backed, 'file', None) is not None:
            backed.file.close()
    return "\n".join(lines)


run_log_path       = output_dir / f"{OUTPUT_PREFIX}_run_log.txt"
structure_log_path = output_dir / f"{OUTPUT_PREFIX}_anndata_structure.txt"

with open(run_log_path, "a", encoding="utf-8") as f:
    f.write("\n".join([
        "=" * 100,
        f"Run timestamp: {datetime.now().isoformat()}",
        f"Reference H5AD: {REFERENCE_H5AD}",
        f"Query H5AD: {QUERY_H5AD}",
        f"Merged output H5AD: {output_h5ad}",
        f"Train output H5AD: {train_h5ad}",
        f"adata_merged shape: {adata_merged.shape}",
        f"adata_train shape: {adata_train.shape}",
    ]) + "\n")

with open(structure_log_path, "w", encoding="utf-8") as f:
    f.write("\n\n".join([
        summarize_anndata_full(adata_merged, "adata_merged (in memory)"),
        summarize_anndata_full(adata_train,  "adata_train (in memory)"),
        summarize_h5ad_on_disk(output_h5ad,  "merged results h5ad"),
        summarize_h5ad_on_disk(train_h5ad,   "train HVG h5ad"),
    ]) + "\n")

print(f"  -> Run log: {run_log_path}")
print(f"  -> AnnData structure summary: {structure_log_path}")


[Step 17b] Writing run log and AnnData structure summary...
  -> Run log: /home/h2048/data/py/20260310/tcell_only_merged_pipeline/tcell_only_merged_run_log.txt
  -> AnnData structure summary: /home/h2048/data/py/20260310/tcell_only_merged_pipeline/tcell_only_merged_anndata_structure.txt


## Cell 21 — Step 18: T Cell Visualization

In [25]:
# STEP_BREAK: c21_viz
print("\n[Step 18] Creating T cell visualizations...")

mask_ref = adata_merged.obs["data_source"] == "reference"
qry_viz  = adata_merged.obs["data_source"] == "query"

fig = plt.figure(figsize=(24, 20))
gs  = fig.add_gridspec(5, 4, hspace=0.3, wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])
sc.pl.umap(adata_merged, color="data_source", ax=ax1, show=False, title="Data Source (scANVI)", s=15)

ax2 = fig.add_subplot(gs[0, 1])
adata_merged.obs["_ref_coarse"] = pd.Series(pd.NA, index=adata_merged.obs_names, dtype="object")
adata_merged.obs.loc[mask_ref, "_ref_coarse"] = adata_merged.obs.loc[mask_ref, "cell_type_coarse"].astype(str).values
adata_merged.obs["_ref_coarse"] = adata_merged.obs["_ref_coarse"].astype("category")
sc.pl.umap(adata_merged, color="_ref_coarse", ax=ax2, show=False, title="Reference Coarse Labels",
           legend_loc="on data", s=15)

ax3 = fig.add_subplot(gs[0, 2])
sc.pl.umap(adata_merged, color="scanvi_pred", ax=ax3, show=False, title="scANVI Predictions",
           legend_loc="on data", s=15)

ax4 = fig.add_subplot(gs[0, 3])
sc.pl.umap(adata_merged, color="scanvi_confidence", ax=ax4, show=False, title="Confidence",
           cmap="viridis", vmin=0, vmax=1, s=15)

ax5 = fig.add_subplot(gs[1, 0])
sc.pl.umap(adata_merged, color="CD4_score", ax=ax5, show=False, title="CD4 Module Score", cmap="Reds", s=15)

ax6 = fig.add_subplot(gs[1, 1])
sc.pl.umap(adata_merged, color="CD8_score", ax=ax6, show=False, title="CD8 Module Score", cmap="Blues", s=15)

ax7 = fig.add_subplot(gs[1, 2])
sc.pl.umap(adata_merged, color="cd4_cd8_by_score", ax=ax7, show=False, title="CD4/CD8 by Score",
           legend_loc="on data", s=15)

ax8 = fig.add_subplot(gs[1, 3])
ax8.scatter(
    adata_merged.obs.loc[qry_viz, "CD4_score"],
    adata_merged.obs.loc[qry_viz, "CD8_score"],
    c=adata_merged.obs.loc[qry_viz, "scanvi_confidence"],
    cmap="viridis", s=5, alpha=0.5
)
ax8.set_xlabel("CD4 Score"); ax8.set_ylabel("CD8 Score")
ax8.set_title(f"Query: CD4 vs CD8 Score (threshold={CD4_SCORE_THRESHOLD})")
ax8.axhline(y=CD8_SCORE_THRESHOLD, color='k', linestyle='--', alpha=0.3)
ax8.axvline(x=CD4_SCORE_THRESHOLD, color='k', linestyle='--', alpha=0.3)

for gene, title, ax_gs in [
    ("CD3E",  "CD3E (pan-T)",          gs[2, 0]),
    ("CD4",   "CD4",                   gs[2, 1]),
    ("CD8A",  "CD8A",                  gs[2, 2]),
    ("CD8B",  "CD8B",                  gs[2, 3]),
    ("CCR7",  "CCR7 (Naive/CM)",       gs[3, 0]),
    ("FOXP3", "FOXP3 (Treg)",          gs[3, 1]),
    ("GZMB",  "GZMB (Effector)",       gs[3, 2]),
    ("MKI67", "MKI67 (Proliferating)", gs[3, 3]),
]:
    ax_i = fig.add_subplot(ax_gs)
    if gene in adata_merged.raw.var_names:
        sc.pl.umap(adata_merged, color=gene, ax=ax_i, show=False, title=title,
                   cmap="Reds", s=15, use_raw=True)

ax17 = fig.add_subplot(gs[4, 0])
ref_c = adata_merged.obs.loc[mask_ref, "cell_type_coarse"].value_counts()
qry_c = adata_merged.obs.loc[qry_viz,  "cd4_cd8_by_score"].value_counts()
x = np.arange(len(ref_c.index)); w = 0.35
ax17.bar(x - w/2, ref_c.values, w, label="Reference (annotated)", alpha=0.8)
ax17.bar(x + w/2, [qry_c.get(k, 0) for k in ref_c.index], w, label="Query (by score)", alpha=0.8)
ax17.set_xticks(x); ax17.set_xticklabels(ref_c.index, rotation=45, ha="right")
ax17.set_ylabel("Cell Count"); ax17.set_title("CD4/CD8 Distribution"); ax17.legend()

ax18 = fig.add_subplot(gs[4, 1])
sc.pl.umap(adata_merged, color="novelty_score", ax=ax18, show=False,
           title="Novelty Score (Query)", cmap="hot", vmin=0, vmax=1, s=15)

ax19 = fig.add_subplot(gs[4, 2])
ax19.hist([adata_merged.obs.loc[mask_ref, "scanvi_confidence"],
           adata_merged.obs.loc[qry_viz,  "scanvi_confidence"]],
          bins=30, label=["Reference", "Query"], alpha=0.7)
ax19.set_xlabel("Confidence"); ax19.set_ylabel("Cell Count")
ax19.set_title("Confidence Distribution by Source"); ax19.legend()

ax20 = fig.add_subplot(gs[4, 3])
sc.pl.umap(adata_merged, color="leiden_query", ax=ax20, show=False,
           title="Query-only Leiden Clusters", legend_loc="on data", s=15)

plt.savefig(output_dir / f"{OUTPUT_PREFIX}_tcell_overview.pdf", dpi=300, bbox_inches="tight")
plt.close()
print(f"  -> Saved: {OUTPUT_PREFIX}_tcell_overview.pdf")

# --- UMAP comparison: scVI vs scANVI ---
# FIX v1.4: use X_umap swap approach.
# sc.pl.umap(basis="X_umap_scVI") is not stable across scanpy versions because
# sc.pl.umap prepends "X_" internally in some releases, causing KeyError or
# plotting the wrong embedding. Safe approach: temporarily swap adata.obsm["X_umap"]
# to the desired space, plot, then restore.
print("  -> Creating scVI vs scANVI UMAP comparison...")
_orig_umap = adata_merged.obsm["X_umap"].copy()

fig2, axes2 = plt.subplots(2, 3, figsize=(18, 12))
color_vars  = ["data_source", "_ref_coarse", "scanvi_pred"]
row_titles  = [
    ["Data Source (scVI UMAP)",   "Reference Labels (scVI UMAP)",   "scANVI Predictions (scVI UMAP)"],
    ["Data Source (scANVI UMAP)", "Reference Labels (scANVI UMAP)", "scANVI Predictions (scANVI UMAP)"],
]
kwargs_map = {
    "data_source": {"legend_loc": "right margin"},
    "_ref_coarse": {"legend_loc": "on data"},
    "scanvi_pred": {"legend_loc": "on data"},
}

for row_i, umap_key in enumerate(["X_umap_scVI", "X_umap_scANVI"]):
    adata_merged.obsm["X_umap"] = adata_merged.obsm[umap_key].copy()
    for col_i, cvar in enumerate(color_vars):
        adata_merged.obs["_ref_coarse"] = adata_merged.obs.get("_ref_coarse", pd.NA)  # ensure present
        sc.pl.umap(adata_merged, color=cvar, ax=axes2[row_i, col_i], show=False,
                   title=row_titles[row_i][col_i], s=10, **kwargs_map[cvar])

adata_merged.obsm["X_umap"] = _orig_umap  # restore default
del _orig_umap

plt.tight_layout()
plt.savefig(output_dir / f"{OUTPUT_PREFIX}_umap_comparison.pdf", dpi=300, bbox_inches="tight")
plt.close()
print(f"  -> Saved: {OUTPUT_PREFIX}_umap_comparison.pdf")

# --- Novelty analysis ---
print("  -> Creating novelty analysis report...")
fig3, axes3 = plt.subplots(2, 2, figsize=(14, 12))

ax = axes3[0, 0]
ax.hist(adata_merged.obs.loc[qry_viz, "novelty_score"], bins=50, edgecolor="black", alpha=0.7)
ax.axvline(x=0.7, color='r', linestyle='--', label='High novelty threshold')
ax.set_xlabel("Novelty Score"); ax.set_ylabel("Cell Count")
ax.set_title("Query Cells: Novelty Score Distribution"); ax.legend()

ax = axes3[0, 1]
sc2 = ax.scatter(
    adata_merged.obs.loc[qry_viz, "scanvi_confidence"],
    adata_merged.obs.loc[qry_viz, "novelty_score"],
    c=adata_merged.obs.loc[qry_viz, "scanvi_entropy"],
    cmap="viridis", s=5, alpha=0.5
)
ax.set_xlabel("scANVI Confidence"); ax.set_ylabel("Novelty Score")
ax.set_title("Query: Confidence vs Novelty"); plt.colorbar(sc2, ax=ax)

ax = axes3[1, 0]
leiden_c = adata_merged.obs.loc[qry_viz, "leiden_query"].value_counts().head(15)
ax.barh(range(len(leiden_c)), leiden_c.values)
ax.set_yticks(range(len(leiden_c))); ax.set_yticklabels(leiden_c.index)
ax.set_xlabel("Cell Count"); ax.set_title("Query-only Leiden Clusters (Top 15)")

ax = axes3[1, 1]
mismatch = []
for idx, row in adata_merged.obs.loc[qry_viz].iterrows():
    st, pred = row["cd4_cd8_by_score"], row["scanvi_pred"]
    if   "CD4" in pred and st == "CD8_single": mismatch.append("PredCD4/ScoreCD8")
    elif "CD8" in pred and st == "CD4_single": mismatch.append("PredCD8/ScoreCD4")
    elif "CD4" in pred and st == "CD4_single": mismatch.append("Match_CD4")
    elif "CD8" in pred and st == "CD8_single": mismatch.append("Match_CD8")
    else:                                        mismatch.append("Other/Unclear")
mc = pd.Series(mismatch).value_counts()
ax.pie(mc.values, labels=mc.index, autopct='%1.1f%%')
ax.set_title("CD4/CD8: Prediction vs Score Agreement")

plt.tight_layout()
plt.savefig(output_dir / f"{OUTPUT_PREFIX}_novelty_analysis.pdf", dpi=300, bbox_inches="tight")
plt.close()
print(f"  -> Saved: {OUTPUT_PREFIX}_novelty_analysis.pdf")

adata_merged.obs.drop(columns=["_ref_coarse"], inplace=True, errors="ignore")

# --- CellTypist direct vs scANVI comparison (NEW in v1.5) ---
ct_panel_cols = [
    k for k in [
        CELLTYPIST_DIRECT_LABEL_KEY,
        CELLTYPIST_DIRECT_FILT_KEY,
        "scanvi_pred",
        f"scanvi_pred_{CELLTYPIST_SCANVI_RESULT_KEY}",
    ]
    if k in adata_merged.obs.columns
]
if len(ct_panel_cols) >= 2:
    n_ct = len(ct_panel_cols)
    fig_ct, axes_ct = plt.subplots(1, n_ct, figsize=(7 * n_ct, 7))
    if n_ct == 1:
        axes_ct = [axes_ct]
    panel_titles = {
        CELLTYPIST_DIRECT_LABEL_KEY:                    "CellTypist Direct",
        CELLTYPIST_DIRECT_FILT_KEY:                     f"CellTypist Filtered (conf>={CELLTYPIST_CONF_THRESHOLD})",
        "scanvi_pred":                                  "scANVI Reference Branch",
        f"scanvi_pred_{CELLTYPIST_SCANVI_RESULT_KEY}":  "scANVI Pseudo-label Branch",
    }
    for ax_ct, col in zip(axes_ct, ct_panel_cols):
        sc.pl.umap(adata_merged, color=col, ax=ax_ct, show=False,
                   title=panel_titles.get(col, col), legend_loc="on data", s=10)
    plt.tight_layout()
    plt.savefig(output_dir / f"{OUTPUT_PREFIX}_celltypist_vs_scanvi_comparison.pdf",
                dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  -> Saved: {OUTPUT_PREFIX}_celltypist_vs_scanvi_comparison.pdf")

# --- Summary ---
qry_mask = adata_merged.obs["data_source"] == "query"
print("\n" + "=" * 80)
print("T CELL PIPELINE COMPLETE (v1.4)")
print("=" * 80)
print(f"Output: {output_dir}")
print(f"  Total cells: {adata_merged.n_obs:,}  |  Reference: {mask_ref.sum():,}  |  Query: {qry_mask.sum():,}")

print("\nAnnotation layers (all cells):")
print(f"  celltypist_label_direct       : pure CellTypist labels")
print(f"  celltypist_label_direct_filt  : same, low-conf (< {CELLTYPIST_CONF_THRESHOLD}) -> Unknown")
print(f"  scanvi_pred_reference         : reference-supervised scANVI (DEFAULT)")
print(f"  scanvi_pred_{CELLTYPIST_SCANVI_RESULT_KEY}: CellTypist pseudo-label scANVI")
print(f"  scanvi_pred                   : alias for reference branch")

print("\nCD4/CD8 Distribution in Query (by score):")
print(adata_merged.obs.loc[qry_mask, "cd4_cd8_by_score"].value_counts())

print("\nTop scANVI predictions in Query:")
print(adata_merged.obs.loc[qry_mask, "scanvi_pred"].value_counts().head(10))

print(f"\nHigh novelty cells (score > 0.7): {adata_merged.obs['is_potentially_novel'].sum():,}")
print(f"Query-only Leiden clusters: {adata_merged.obs.loc[qry_mask, 'leiden_query'].nunique()}")

print("\nUMAP spaces: X_umap (DEFAULT/scANVI) | X_umap_scVI | X_umap_scANVI")

print("\n" + "=" * 80)
print("Bug Fixes in v1.4 (incremental from v1.3):")
print("  BUG7  [P1] ensure_batch_tissue: batch_key also fillna")
print("  BUG8  [P1] BATCH_KEY prefixed ref_/qry_ before concat")
print("  BUG9  [P0] export_celltypist_direct_results: first-class CellTypist output")
print("  BUG10 [P1] UMAP comparison: X_umap swap instead of basis= kwarg")
print("  BUG11 [P2] Removed unused hvg_mask param")
print("=" * 80)


[Step 18] Creating T cell visualizations...
  -> Saved: tcell_only_merged_tcell_overview.pdf
  -> Creating scVI vs scANVI UMAP comparison...
  -> Saved: tcell_only_merged_umap_comparison.pdf
  -> Creating novelty analysis report...
  -> Saved: tcell_only_merged_novelty_analysis.pdf
  -> Saved: tcell_only_merged_celltypist_vs_scanvi_comparison.pdf

T CELL PIPELINE COMPLETE (v1.4)
Output: /home/h2048/data/py/20260310/tcell_only_merged_pipeline
  Total cells: 190,168  |  Reference: 44,833  |  Query: 145,335

Annotation layers (all cells):
  celltypist_label_direct       : pure CellTypist labels
  celltypist_label_direct_filt  : same, low-conf (< 0.5) -> Unknown
  scanvi_pred_reference         : reference-supervised scANVI (DEFAULT)
  scanvi_pred_celltypist_pseudolabel: CellTypist pseudo-label scANVI
  scanvi_pred                   : alias for reference branch

CD4/CD8 Distribution in Query (by score):
cd4_cd8_by_score
CD4_single    79532
DN            65803
Name: count, dtype: int64

Top

## Cell 22 — End of Pipeline

In [26]:
# Pipeline complete. All output files are in OUTPUT_DIR.
# To reload results:
#   adata = sc.read_h5ad(output_h5ad)
print(f"All outputs saved to: {output_dir}")

All outputs saved to: /home/h2048/data/py/20260310/tcell_only_merged_pipeline
